In [ ]:
real_images = []
real_jsons = []
real_txts = []
real_xmls = []


# SIH26247 — Step 1 Part 2
## Pass 2 → Pass 3 → Pass 4

This notebook continues the **already completed Step 1 Pass 1** checkpoint.

```text
PASS 1 — Seraphim
        ↓
step1_pass1_rtdetr_cradio_v3_l.pth
        ↓
PASS 2 — RealDroneVision
        ↓
step1_pass2_rtdetr_cradio_v3_l.pth
        ↓
PASS 3 — DroneDetectionThesis RGB
        ↓
step1_pass3_rtdetr_cradio_v3_l.pth
        ↓
PASS 4 — UAV-CB RGB
        ↓
FINAL STEP 1 RGB MODEL
```

Frozen model choice:

**NVIDIA TAO RT-DETR + C-RADIOv3-L**

The previous RT-DETR checkpoint is passed through `train.pretrained_model_path`.
The C-RADIOv3-L foundation checkpoint remains in `model.frozen_fm.checkpoint` for every pass.

This notebook is deliberately **training-first**:
- no full test-set inference;
- no massive prediction files;
- no ONNX export;
- no TensorRT export;
- no YOLO-style custom inference;
- only the small validation data TAO needs to initialize training.

## Important Colab / TAO 7.2 note

Do **not** try to install Docker inside a normal hosted Google Colab runtime for this project.

NVIDIA's current TAO 7.2 release is built around the TAO Execution SDK and NVIDIA containers. The current PyTorch training image is:

```text
nvcr.io/nvidia/tao/tao-toolkit:7.2.0-pyt
```

NVIDIA's old documentation had a Google-Colab TAO workflow, but NVIDIA's TAO repository marks TAO-via-Colab as deprecated from 6.0 onward. The current 7.2 documentation describes local Docker and managed GPU backends instead.

Therefore this notebook has two modes:

```text
MANAGED_COLAB
    ↓
download / prepare datasets
    ↓
create COCO + TAO configs
    ↓
save artifacts to Drive
    ↓
run training on a Docker-capable NVIDIA machine

LOCAL_RUNTIME
    ↓
Colab frontend connected to your own Windows/NVIDIA machine
    ↓
use its existing Docker + GPU
    ↓
run TAO 7.2 directly
```

If you use ordinary hosted Colab, the data-preparation cells work. The TAO training cell stops with a clear message instead of pretending that Docker is available.

In [1]:
# ============================================================
# 0. ENVIRONMENT + MODE
# ============================================================

import os
import sys
import json
import random
import shutil
import subprocess
import platform
import zipfile
import urllib.request
import re
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()
IS_WINDOWS = os.name == "nt"
IS_LINUX = sys.platform.startswith("linux")

# ------------------------------------------------------------
# CHOOSE MODE
# ------------------------------------------------------------
# "AUTO"       = hosted Colab -> MANAGED_COLAB; local Jupyter -> LOCAL_RUNTIME
# "MANAGED_COLAB" = data preparation in hosted Colab
# "LOCAL_RUNTIME" = training through your own Docker-capable NVIDIA host
MODE = "AUTO"

if MODE == "AUTO":
    MODE = "MANAGED_COLAB" if IS_COLAB and not IS_WINDOWS else "LOCAL_RUNTIME"

print("Platform:", platform.platform())
print("Python  :", sys.version.split()[0])
print("IS_COLAB:", IS_COLAB)
print("MODE    :", MODE)


Platform: Windows-10-10.0.26300-SP0
Python  : 3.9.13
IS_COLAB: True
MODE    : LOCAL_RUNTIME


In [2]:
# ============================================================
# 1. WORKSPACE PATHS
# ============================================================

TAO_IMAGE = "nvcr.io/nvidia/tao/tao-toolkit:7.2.0-pyt"

if MODE == "MANAGED_COLAB":
    WORKSPACE = Path(r"H:\Hobby Projrctrs\Sih26change")
    DRIVE_ROOT = Path(r"F:\Sih26change_Data\data\step1\drive_mock")
else:
    # Your existing Windows paths.
    PROJECT_ROOT = Path(r"H:\Hobby Projrctrs\Sih26change")
    DATA_ROOT = Path(r"F:\Sih26change_Data\data")
    RUNS_ROOT = Path(r"F:\Sih26change_Data\runs")
    MODELS_ROOT = Path(r"F:\Sih26change_Data\models")

    WORKSPACE = PROJECT_ROOT
    DRIVE_ROOT = None

if MODE == "MANAGED_COLAB":
    DATA_ROOT = Path(r"F:\Sih26change_Data\data")
    RUNS_ROOT = WORKSPACE / "runs"
    MODELS_ROOT = WORKSPACE / "models"

CONFIG_ROOT = WORKSPACE / "configs"
WEIGHTS_ROOT = WORKSPACE / "weights"
STEP1_DATA = DATA_ROOT / "step1"
STEP1_MODELS = MODELS_ROOT / "step1"

for p in [
    WORKSPACE, DATA_ROOT, RUNS_ROOT, MODELS_ROOT,
    CONFIG_ROOT, WEIGHTS_ROOT, STEP1_DATA, STEP1_MODELS
]:
    p.mkdir(parents=True, exist_ok=True)

print("WORKSPACE :", WORKSPACE)
print("DATA_ROOT :", DATA_ROOT)
print("MODELS    :", MODELS_ROOT)
print("CONFIGS   :", CONFIG_ROOT)


WORKSPACE : H:\Hobby Projrctrs\Sih26change
DATA_ROOT : F:\Sih26change_Data\data
MODELS    : F:\Sih26change_Data\models
CONFIGS   : H:\Hobby Projrctrs\Sih26change\configs


In [3]:
# ============================================================
# GOOGLE DRIVE CELL BYPASSED FOR LOCAL EXECUTION
# ============================================================
print('Google Drive logic bypassed.')


Google Drive logic bypassed.


In [4]:
# ============================================================
# 3. FIND / COPY PASS-1 CHECKPOINT
# ============================================================

PASS1_NAME = "step1_pass1_rtdetr_cradio_v3_l.pth"

def find_named(root, filename):
    hits = []
    if root and Path(root).exists():
        hits = list(Path(root).rglob(filename))
    return hits

pass1_candidates = []

if MODE == "MANAGED_COLAB":
    # Search both the uploaded files in /content and Drive
    pass1_candidates.extend(
        find_named("/content", PASS1_NAME)
    )
    pass1_candidates.extend(
        find_named("/content/drive/MyDrive", PASS1_NAME)
    )
else:
    # Search the model store and project workspace.
    pass1_candidates.extend(
        find_named(MODELS_ROOT, PASS1_NAME)
    )
    pass1_candidates.extend(
        find_named(WORKSPACE, PASS1_NAME)
    )

# Remove duplicates while preserving order
pass1_candidates = list(dict.fromkeys(pass1_candidates))

print("Pass-1 checkpoint candidates:")
for p in pass1_candidates[:20]:
    print(" ", p)

if not pass1_candidates:
    raise FileNotFoundError(
        f"Could not find {PASS1_NAME}. "
        "In managed Colab, upload/copy the Pass-1 model into Google Drive or /content first."
    )

PASS1_SOURCE = pass1_candidates[0]
PASS1_LOCAL = STEP1_MODELS / PASS1_NAME

if PASS1_SOURCE.resolve() != PASS1_LOCAL.resolve():
    shutil.copy2(PASS1_SOURCE, PASS1_LOCAL)

print("Pass-1 checkpoint available at:")
print(PASS1_LOCAL)
print("Size GB:", round(PASS1_LOCAL.stat().st_size / 1024**3, 3))


Pass-1 checkpoint candidates:
  F:\Sih26change_Data\models\step1\step1_pass1_rtdetr_cradio_v3_l.pth
Pass-1 checkpoint available at:
F:\Sih26change_Data\models\step1\step1_pass1_rtdetr_cradio_v3_l.pth
Size GB: 1.39


# PASS 2 — RealDroneVision

RealDroneVision is the large real-world drone dataset introduced at WACV 2026. The official paper reports **173,023 images**, and its supplementary material provides the official Google Drive dataset link.

For this training-first workflow:

- use the RealDroneVision training data;
- create a modest validation subset from that training pool only when the archive does not already provide a clean validation split;
- do not process the full held-out test set;
- do not generate full prediction files.

In [5]:
# ============================================================
# 4. PASS 2 — DOWNLOAD REALDRONEVISION
# ============================================================

REAL_ROOT = STEP1_DATA / "realdronevision"
REAL_ROOT.mkdir(parents=True, exist_ok=True)

REAL_GDRIVE_ID = "1ftZ8It_x1IOQ0qH_Y1GPh5Ug5yDWinGp"

# Check if it is already present.
existing_images = [
    p for p in REAL_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp"}
]

if not existing_images:
    print("Installing gdown...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "gdown"],
        check=True
    )

    import gdown

    DOWNLOAD_DIR = REAL_ROOT / "download"
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

    print("Downloading RealDroneVision from the official dataset folder.")
    print("The official WACV supplementary material links this Google Drive folder.")

    try:
        gdown.download_folder(
            id=REAL_GDRIVE_ID,
            output=str(DOWNLOAD_DIR),
            quiet=False,
            use_cookies=False
        )
    except Exception as e:
        print("\n[WARNING] Automated folder download failed.")
        print("Google Drive may have flagged this folder due to high traffic (404/403 limit).")
        print(f"Error details: {e}")
        print("\nMANUAL STEP REQUIRED:")
        print(f"1. Open the Google Drive folder ID: {REAL_GDRIVE_ID} in your browser.")
        print("2. Download the contents (images & annotations) as a ZIP archive.")
        print(f"3. Upload/copy the files directly into: {REAL_ROOT}")
        print("4. Once uploaded, re-run this notebook's cells.")

print("RealDroneVision files now visible under:")
print(REAL_ROOT)


RealDroneVision files now visible under:
F:\Sih26change_Data\data\step1\realdronevision


In [7]:
# ============================================================
# 6. PASS 2 — GENERIC ANNOTATION DETECTION
# ============================================================

import xml.etree.ElementTree as ET
from PIL import Image

def load_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8",
            errors="replace"
        )
    )

def is_coco_json(path):
    try:
        d = load_json(path)
        return (
            isinstance(d, dict)
            and "images" in d
            and "annotations" in d
            and "categories" in d
        )
    except Exception:
        return False

coco_jsons = [p for p in real_jsons if is_coco_json(p)]

print("COCO JSON files detected:", len(coco_jsons))

def classify_split(path):
    parts = {x.lower() for x in path.parts}

    if "train" in parts or "training" in parts:
        return "train"

    if "test" in parts or "testing" in parts:
        return "test"

    if "val" in parts or "valid" in parts or "validation" in parts:
        return "val"

    return "unknown"

split_counts = {"train":0, "val":0, "test":0, "unknown":0}
for p in real_images:
    split_counts[classify_split(p)] += 1

print("Detected image splits:", split_counts)


COCO JSON files detected: 0
Detected image splits: {'train': 0, 'val': 0, 'test': 0, 'unknown': 5537}


### RealDroneVision annotation handling

The dataset was published for object detection and its official training experiments use 640×640 image training. Because the public Drive archive can change its exact folder packaging, this notebook does not silently assume one annotation layout.

The next cell searches for, in order:

1. a COCO annotation file;
2. per-image YOLO `.txt` annotations;
3. LabelMe-style JSON;
4. Pascal VOC XML.

If the archive uses another layout, the notebook stops at the annotation-detection stage instead of silently generating incorrect labels.

In [8]:
# ============================================================
# 7. PASS 2 — RESOLVE ANNOTATION MODE
# ============================================================

ANNOTATION_MODE = None
COCO_SOURCE = None

if coco_jsons:
    ANNOTATION_MODE = "COCO"
    # Prefer a file whose path contains train.
    train_candidates = [
        p for p in coco_jsons
        if "train" in {x.lower() for x in p.parts}
        or "training" in {x.lower() for x in p.parts}
    ]
    COCO_SOURCE = train_candidates[0] if train_candidates else coco_jsons[0]

elif real_txts:
    # Check whether image stems have matching text files.
    match = sum(
        1 for img in real_images
        if img.with_suffix(".txt").exists()
    )
    if match > 0:
        ANNOTATION_MODE = "YOLO"

elif real_jsons:
    # Check for LabelMe-style JSON.
    match = 0
    for img in real_images[:100]:
        jp = img.with_suffix(".json")
        if not jp.exists():
            continue
        try:
            d = load_json(jp)
            if "shapes" in d:
                match += 1
        except Exception:
            pass

    if match > 0:
        ANNOTATION_MODE = "LABELME"

elif real_xmls:
    match = sum(
        1 for img in real_images
        if img.with_suffix(".xml").exists()
    )
    if match > 0:
        ANNOTATION_MODE = "VOC"

print("Resolved annotation mode:", ANNOTATION_MODE)
if COCO_SOURCE:
    print("COCO source:", COCO_SOURCE)

if ANNOTATION_MODE is None:
    raise RuntimeError(
        "Could not automatically identify the RealDroneVision annotation layout. "
        "Inspect the files printed above before training."
    )


Resolved annotation mode: LABELME


In [9]:
# ============================================================
# 8. PASS 2 — MAKE TRAIN/VAL COCO WITHOUT FULL TEST
# ============================================================

REAL_COCO = REAL_ROOT / "tao_coco"
REAL_TRAIN = REAL_COCO / "train"
REAL_VAL = REAL_COCO / "val"

for p in [
    REAL_TRAIN / "images",
    REAL_VAL / "images",
]:
    p.mkdir(parents=True, exist_ok=True)

def yolo_boxes(label_path, width, height):
    boxes = []
    for line in Path(label_path).read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines():

        parts = line.strip().split()
        if len(parts) < 5:
            continue

        _, xc, yc, bw, bh = map(float, parts[:5])

        x = (xc - bw/2) * width
        y = (yc - bh/2) * height
        w = bw * width
        h = bh * height

        boxes.append([x,y,w,h])

    return boxes

def labelme_boxes(label_path):
    d = load_json(label_path)
    boxes = []

    for shape in d.get("shapes", []):
        label = str(shape.get("label","")).strip().lower()

        if label not in {"drone","uav"}:
            continue

        pts = shape.get("points", [])
        if len(pts) < 2:
            continue

        xs = [float(p[0]) for p in pts]
        ys = [float(p[1]) for p in pts]

        x1,x2 = min(xs),max(xs)
        y1,y2 = min(ys),max(ys)

        boxes.append([x1,y1,x2-x1,y2-y1])

    return boxes

def voc_boxes(label_path):
    root = ET.parse(label_path).getroot()
    boxes = []

    for obj in root.findall("object"):
        label = obj.findtext("name", default="drone").strip().lower()

        if label not in {"drone","uav"}:
            continue

        b = obj.find("bndbox")

        xmin = float(b.findtext("xmin"))
        ymin = float(b.findtext("ymin"))
        xmax = float(b.findtext("xmax"))
        ymax = float(b.findtext("ymax"))

        boxes.append([
            xmin,
            ymin,
            xmax-xmin,
            ymax-ymin
        ])

    return boxes

def sidecar_boxes(img):
    with Image.open(img) as im:
        w,h = im.size

    if ANNOTATION_MODE == "YOLO":
        return yolo_boxes(img.with_suffix(".txt"), w, h)

    if ANNOTATION_MODE == "LABELME":
        return labelme_boxes(img.with_suffix(".json"))

    if ANNOTATION_MODE == "VOC":
        return voc_boxes(img.with_suffix(".xml"))

    raise RuntimeError("COCO is handled separately.")

# Select train images.
if ANNOTATION_MODE == "COCO":
    coco_data = load_json(COCO_SOURCE)

    # Resolve COCO image -> absolute source image.
    image_lookup = {}
    for img_entry in coco_data["images"]:
        fn = img_entry["file_name"]
        matches = list(REAL_ROOT.rglob(Path(fn).name))
        if matches:
            image_lookup[img_entry["id"]] = matches[0]

    # Create image -> boxes from COCO.
    boxes_by_image = {}
    for ann in coco_data["annotations"]:
        boxes_by_image.setdefault(
            ann["image_id"], []
        ).append(
            ann["bbox"]
        )

    coco_pairs = []
    for img_id, src in image_lookup.items():
        coco_pairs.append(
            (src, boxes_by_image.get(img_id, []))
        )

    train_pairs = [
        x for x in coco_pairs
        if classify_split(x[0]) != "test"
    ]

else:
    # Use images with matching sidecar annotations.
    matched = []

    for img in real_images:
        if ANNOTATION_MODE == "YOLO":
            label = img.with_suffix(".txt")
        elif ANNOTATION_MODE == "LABELME":
            label = img.with_suffix(".json")
        elif ANNOTATION_MODE == "VOC":
            label = img.with_suffix(".xml")
        else:
            label = None

        if label and label.exists():
            matched.append(img)

    train_pool = [
        img for img in matched
        if classify_split(img) != "test"
    ]

    train_pairs = [
        (img, None)
        for img in train_pool
    ]

print("Training pool before validation holdout:", len(train_pairs))

assert train_pairs, "No annotated RealDroneVision training images found."

# Small 2% validation subset from the training pool.
rng = random.Random(42)
rng.shuffle(train_pairs)

val_count = max(
    1000,
    int(0.02 * len(train_pairs))
)

# Keep it capped to avoid unnecessary validation storage.
val_count = min(val_count, 5000)

val_pairs = train_pairs[:val_count]
train_pairs = train_pairs[val_count:]

print("Pass-2 training images:", len(train_pairs))
print("Pass-2 validation images:", len(val_pairs))


Training pool before validation holdout: 5530
Pass-2 training images: 4530
Pass-2 validation images: 1000


In [10]:
# ============================================================
# 9. PASS 2 — WRITE COCO
# ============================================================

def build_coco_pairs(pairs, image_out, json_out, description):
    image_out.mkdir(parents=True, exist_ok=True)

    images = []
    annotations = []
    ann_id = 1

    for image_id, pair in enumerate(pairs, start=1):
        src, predefined_boxes = pair

        dst = image_out / src.name

        if dst.exists():
            dst = image_out / f"{src.parent.name}__{src.name}"

        shutil.copy2(src, dst)

        with Image.open(src) as im:
            width,height = im.size

        images.append({
            "id": image_id,
            "file_name": dst.name,
            "width": width,
            "height": height
        })

        if predefined_boxes is not None:
            boxes = predefined_boxes
        else:
            boxes = sidecar_boxes(src)

        for box in boxes:
            x,y,w,h = map(float, box)

            x = max(0,min(x,width))
            y = max(0,min(y,height))
            w = max(0,min(w,width-x))
            h = max(0,min(h,height-y))

            if w <= 0 or h <= 0:
                continue

            annotations.append({
                "id": ann_id,
                "image_id": image_id,
                "category_id": 1,
                "bbox": [x,y,w,h],
                "area": w*h,
                "iscrowd": 0
            })

            ann_id += 1

    payload = {
        "info": {"description": description},
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": [
            {"id":1,"name":"drone"}
        ]
    }

    json_out.write_text(
        json.dumps(payload),
        encoding="utf-8"
    )

    return len(images),len(annotations)

pass2_train_stats = build_coco_pairs(
    train_pairs,
    REAL_TRAIN / "images",
    REAL_TRAIN / "annotations.json",
    "SIH26247 Step 1 Pass 2 — RealDroneVision training"
)

pass2_val_stats = build_coco_pairs(
    val_pairs,
    REAL_VAL / "images",
    REAL_VAL / "annotations.json",
    "SIH26247 Step 1 Pass 2 — RealDroneVision small validation"
)

print("Train:", pass2_train_stats)
print("Val  :", pass2_val_stats)


Train: (4530, 4530)
Val  : (1000, 1000)


In [11]:
# ============================================================
# 10. PASS 2 — COCO INTEGRITY CHECK
# ============================================================

def check_coco(json_path, image_dir):
    d = json.loads(
        Path(json_path).read_text(
            encoding="utf-8"
        )
    )

    assert d["categories"] == [
        {"id":1,"name":"drone"}
    ]

    image_ids = {x["id"] for x in d["images"]}

    for ann in d["annotations"]:
        assert ann["image_id"] in image_ids
        assert ann["category_id"] == 1

    missing = [
        x["file_name"]
        for x in d["images"]
        if not (Path(image_dir)/x["file_name"]).exists()
    ]

    assert not missing, missing[:5]

    print(
        Path(json_path).name,
        "images =", len(d["images"]),
        "boxes =", len(d["annotations"])
    )

check_coco(
    REAL_TRAIN / "annotations.json",
    REAL_TRAIN / "images"
)

check_coco(
    REAL_VAL / "annotations.json",
    REAL_VAL / "images"
)


annotations.json images = 4530 boxes = 4530


annotations.json images = 1000 boxes = 1000


# Pass 2 TAO specification

The critical checkpoint relationship is:

```text
model.frozen_fm.checkpoint
        =
C-RADIOv3-L foundation

train.pretrained_model_path
        =
Pass-1 RT-DETR checkpoint
```

The foundation checkpoint is **not replaced** by the Pass-1 checkpoint.

In [12]:
# ============================================================
# 11. PASS 2 — TAO YAML
# ============================================================

PASS2_SPEC = CONFIG_ROOT / "rtdetr_step1_pass2_realdronevision.yaml"

PASS1_CONTAINER = "/workspace/models/step1/step1_pass1_rtdetr_cradio_v3_l.pth"

pass2_yaml = f"""
results_dir: /workspace/runs/step1_pass2

model:
  frozen_fm:
    enabled: true
    backbone: radio_v3-l
    checkpoint: /workspace/weights/c-radio-v3_l_half.pth.tar

  num_queries: 300
  num_select: 300
  num_feature_levels: 3
  return_interm_indices: [1, 2, 3]
  feat_strides: [8, 16, 32]
  feat_channels: [256, 256, 256]
  use_encoder_idx: [2]
  hidden_dim: 256
  nheads: 8
  dropout_ratio: 0.0
  enc_layers: 1
  dim_feedforward: 1024
  pe_temperature: 10000
  expansion: 1.0
  depth_mult: 1
  enc_act: gelu
  act: silu
  dec_layers: 6
  dn_number: 100
  eval_idx: -1

  vfl_loss_coef: 1.0
  bbox_loss_coef: 5.0
  giou_loss_coef: 2.0
  alpha: 0.75
  gamma: 2.0

  aux_loss: true
  loss_types: [vfl, boxes]

dataset:
  train_data_sources:
    - image_dir: /workspace/data/step1/realdronevision/tao_coco/train/images
      json_file: /workspace/data/step1/realdronevision/tao_coco/train/annotations.json

  val_data_sources:
    image_dir: /workspace/data/step1/dronedetectionthesis_rgb/tao_coco/val/images
    json_file: /workspace/data/step1/dronedetectionthesis_rgb/tao_coco/val/annotations.json

  num_classes: 2
  eval_class_ids: [1]

  batch_size: 2
  workers: 4
  pin_memory: true
  dataset_type: serialized

  augmentation:
    multi_scales: [640]
    train_spatial_size: [640, 640]
    eval_spatial_size: [640, 640]
    distortion_prob: 0.0
    iou_crop_prob: 0.8
    preserve_aspect_ratio: true

train:
  num_gpus: 1
  gpu_ids: [0]
  num_nodes: 1
  distributed_strategy: ddp

  num_epochs: 5
  validation_interval: 10
  checkpoint_interval: 10

  clip_grad_norm: 0.1
  precision: fp16
  activation_checkpoint: true
  enable_ema: true

  pretrained_model_path: {PASS1_CONTAINER}

  optim:
    optimizer: AdamW
    lr: 0.00005
    lr_backbone: 0.000005
    weight_decay: 0.0001
    lr_scheduler: MultiStep
    lr_decay: 0.1
    lr_steps: [70]
"""

PASS2_SPEC.write_text(
    pass2_yaml.strip() + "\n",
    encoding="utf-8"
    )



print(PASS2_SPEC)


H:\Hobby Projrctrs\Sih26change\configs\rtdetr_step1_pass2_realdronevision.yaml


# Pass 2 training

For a **managed Colab runtime**, stop here after preparing the dataset/config and copy the prepared workspace to your chosen NVIDIA Docker host.

For a **local runtime connected to the Windows GPU machine**, the training cell can call your existing Docker installation directly.

In [13]:
# ============================================================
# 12. COMMON DOCKER RUNNER
# ============================================================

def run_cmd(cmd, check=True):
    print("$", cmd)

    p = subprocess.run(
        cmd,
        shell=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    print(p.stdout)

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {p.returncode}"
        )

    return p

def docker_available():
    return shutil.which("docker") is not None

print("Docker available:", docker_available())


Docker available: True


In [14]:
print("Pass 2 docker run skipped")


$ docker run --rm --gpus all --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 -v "H:\Hobby Projrctrs\Sih26change:/workspace" nvcr.io/nvidia/tao/tao-toolkit:7.2.0-pyt rtdetr train -e /workspace/configs/rtdetr_step1_pass2_realdronevision_dryrun.yaml



=== TAO Toolkit PyTorch ===

NVIDIA Release 7.2.0-PyTorch (build 286725635)
TAO Toolkit Version 7.2.0

Various files include modifications (c) NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

This container image and its contents are governed by the TAO Toolkit End User License Agreement.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/tao-toolkit-software-license-agreement

  Using driver version 592.82 which has support for CUDA 13.1.  This container
  was built with CUDA 13.2 and will be run in Minor Version Compatibility mode.
  CUDA Forward Compatibility is preferred over Minor Version Compatibility for use
  with this container but was unavailable:
  [[]]
  See https://docs.nvidia.com/deploy/cuda-compatibility/ for details.

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.w

In [15]:
print("Docker run disabled by script")


$ docker run --rm --gpus all --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 -v "H:\Hobby Projrctrs\Sih26change:/workspace" nvcr.io/nvidia/tao/tao-toolkit:7.2.0-pyt rtdetr train -e /workspace/configs/rtdetr_step1_pass2_realdronevision.yaml



=== TAO Toolkit PyTorch ===

NVIDIA Release 7.2.0-PyTorch (build 286725635)
TAO Toolkit Version 7.2.0

Various files include modifications (c) NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

This container image and its contents are governed by the TAO Toolkit End User License Agreement.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/tao-toolkit-software-license-agreement

  Using driver version 592.82 which has support for CUDA 13.1.  This container
  was built with CUDA 13.2 and will be run in Minor Version Compatibility mode.
  CUDA Forward Compatibility is preferred over Minor Version Compatibility for use
  with this container but was unavailable:
  [[]]
  See https://docs.nvidia.com/deploy/cuda-compatibility/ for details.

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.w

In [16]:
# ============================================================
# 15. PASS 2 — SAVE CHECKPOINT
# ============================================================

PASS2_RUN = RUNS_ROOT / "step1_pass2"

ckpts = sorted(
    PASS2_RUN.rglob("model_epoch_*.pth")
)

print("Pass-2 checkpoints:", len(ckpts))

if ckpts:
    PASS2_FINAL = STEP1_MODELS / "step1_pass2_rtdetr_cradio_v3_l.pth"

    shutil.copy2(
        ckpts[-1],
        PASS2_FINAL
    )

    print("PASS 2 COMPLETE:")
    print(PASS2_FINAL)

    if MODE == "MANAGED_COLAB":
        drive_dest = DRIVE_STEP1_MODELS / PASS2_FINAL.name
        shutil.copy2(PASS2_FINAL, drive_dest)
        print("Copied to Google Drive:")
        print(drive_dest)
else:
    print("No Pass-2 checkpoint yet. Complete Pass 2 on the NVIDIA training host.")


Pass-2 checkpoints: 0
No Pass-2 checkpoint yet. Complete Pass 2 on the NVIDIA training host.


# PASS 3 — DroneDetectionThesis RGB

Only the **visible drone sequences** are used for this RGB pass.

The official repository contains visible video and label data in `Data/Video_V`. The same source-video-aware split principle is retained: a source sequence cannot contribute frames to multiple splits.

This notebook again avoids the full test set. A small validation set is created for training-time dataloader compatibility.

In [17]:
# ============================================================
# 16. PASS 3 - DDT RGB DATA (BYPASSED CLONE)
# ============================================================
import os
from pathlib import Path
WORKSPACE = Path(r"H:\Hobby Projrctrs\Sih26change")
DATA_ROOT = Path(r"F:\Sih26change_Data\data")
STEP1_DATA = DATA_ROOT / "step1"
DDT_ROOT = STEP1_DATA / "dronedetectionthesis_rgb"
DDT_EXTRACT = DDT_ROOT / "download" / "extracted"
print('Bypassing Git clone and MP4 extraction. Using pre-extracted files.')


Bypassing Git clone and MP4 extraction. Using pre-extracted files.


In [18]:
# ============================================================
# 16b. PASS 3 - VERIFY DDT (BYPASSED)
# ============================================================
print('Verify bypassed.')


Verify bypassed.


In [19]:
print('Decoder import bypassed.')


Decoder import bypassed.

In [20]:
print('Decoder import bypassed.')


Decoder import bypassed.


In [21]:
# ============================================================
# 18. PASS 3 — DIAGNOSTIC SCAN OF THE WORKSPACE
# ============================================================

from pathlib import Path
import os

print("Current working directory:", os.getcwd())
print("List of files in /content:")
for p in Path("/content").glob("**/*"):
    if p.is_dir() and "packages" not in p.parts and "drive" not in p.parts:
        print(" Dir: ", p)
    elif p.is_file() and "packages" not in p.parts and "drive" not in p.parts:
        # Limit depth representation
        if len(p.parts) <= 5:
            print(" File:", p)


Current working directory: H:\Hobby Projrctrs\Sih26change
List of files in /content:
 Dir:  \content\SIH26247


 Dir:  \content\SIH26247\configs
 Dir:  \content\SIH26247\data
 Dir:  \content\SIH26247\models
 Dir:  \content\SIH26247\runs
 File: \content\SIH26247\configs\rtdetr_step1_pass3_dronedetectionthesis_rgb.yaml
 File: \content\SIH26247\configs\rtdetr_step1_pass3_dryrun.yaml
 Dir:  \content\SIH26247\data\step1
 Dir:  \content\SIH26247\data\step1\dronedetectionthesis_rgb
 Dir:  \content\SIH26247\data\step1\dronedetectionthesis_rgb\Drone-detection-dataset
 Dir:  \content\SIH26247\data\step1\dronedetectionthesis_rgb\Drone-detection-dataset\.git
 Dir:  \content\SIH26247\data\step1\dronedetectionthesis_rgb\Drone-detection-dataset\.git\hooks
 Dir:  \content\SIH26247\data\step1\dronedetectionthesis_rgb\Drone-detection-dataset\.git\info
 Dir:  \content\SIH26247\data\step1\dronedetectionthesis_rgb\Drone-detection-dataset\.git\objects
 Dir:  \content\SIH26247\data\step1\dronedetectionthesis_rgb\Drone-detection-dataset\.git\refs


 Dir: 

 \content\SIH26247\data\step1\dronedetectionthesis_rgb\Drone-detection-dataset\.git\objects\info
 Dir:  \content\SIH26247\data\step1\dronedetectionthesis_rgb\Drone-detection-dataset\.git\objects\pack
 Dir:  \content\SIH26247\data\step1\dronedetectionthesis_rgb\Drone-detection-dataset\.git\refs\heads
 Dir:  \content\SIH26247\data\step1\dronedetectionthesis_rgb\Drone-detection-dataset\.git\refs\tags
 Dir:  \content\SIH26247\models\step1


In [22]:
# ============================================================
# EXTRACTION BYPASSED
# ============================================================
print('Extraction bypassed since files are already present.')


Extraction bypassed since files are already present.


In [23]:
from pathlib import Path

DDT_ROOT = Path("/content/SIH26247/data/step1/dronedetectionthesis_rgb")
DDT_EXTRACT = DDT_ROOT / "extracted"

print("DDT_ROOT:")
print(DDT_ROOT)

print("\nDDT_EXTRACT:")
print(DDT_EXTRACT)

for split in ["train", "val", "test"]:
    folder = DDT_EXTRACT / split

    jpgs = list(folder.glob("*.jpg"))
    jsons = list(folder.glob("*.json"))

    print(f"\n{split.upper()}")
    print("Folder exists :", folder.exists())
    print("JPG files     :", len(jpgs))
    print("JSON files    :", len(jsons))

    if jpgs:
        print("Example JPG   :", jpgs[0])

    if jsons:
        print("Example JSON  :", jsons[0])


DDT_ROOT:


\content\SIH26247\data\step1\dronedetectionthesis_rgb

DDT_EXTRACT:
\content\SIH26247\data\step1\dronedetectionthesis_rgb\extracted

TRAIN
Folder exists : False
JPG files     : 0
JSON files    : 0

VAL
Folder exists : False
JPG files     : 0
JSON files    : 0

TEST
Folder exists : False
JPG files     : 0
JSON files    : 0


In [24]:
# ============================================================
# PASS 3 — DDT RGB TAO YAML
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# 1. LOCAL CONFIG PATH
# ------------------------------------------------------------
WORKSPACE = Path(r"H:\Hobby Projrctrs\Sih26change")
CONFIG_ROOT = WORKSPACE / "configs"
CONFIG_ROOT.mkdir(parents=True, exist_ok=True)

PASS3_SPEC = CONFIG_ROOT / "rtdetr_step1_pass3_dronedetectionthesis_rgb.yaml"

# ------------------------------------------------------------
# 2. TAO CONTAINER PATHS
# ------------------------------------------------------------
PASS2_CONTAINER = "/workspace/models/step1/step1_pass2_rtdetr_cradio_v3_l.pth"

DDT_TRAIN = "/workspace/data/step1/dronedetectionthesis_rgb/tao_coco/train"
DDT_VAL   = "/workspace/data/step1/dronedetectionthesis_rgb/tao_coco/val"

# ------------------------------------------------------------
# 3. BUILD PASS 3 YAML DIRECTLY
# ------------------------------------------------------------
pass3_yaml = f"""
results_dir: /workspace/runs/step1_pass3

model:
  frozen_fm:
    enabled: true
    backbone: radio_v3-l
    checkpoint: /workspace/weights/c-radio-v3_l_half.pth.tar

  num_queries: 300
  num_select: 300
  num_feature_levels: 3
  return_interm_indices: [1, 2, 3]
  feat_strides: [8, 16, 32]
  feat_channels: [256, 256, 256]
  use_encoder_idx: [2]
  hidden_dim: 256
  nheads: 8
  dropout_ratio: 0.0
  enc_layers: 1
  dim_feedforward: 1024
  pe_temperature: 10000
  expansion: 1.0
  depth_mult: 1
  enc_act: gelu
  act: silu
  dec_layers: 6
  dn_number: 100
  eval_idx: -1

  vfl_loss_coef: 1.0
  bbox_loss_coef: 5.0
  giou_loss_coef: 2.0
  alpha: 0.75
  gamma: 2.0

  aux_loss: true
  loss_types: [vfl, boxes]

dataset:
  train_data_sources:
    - image_dir: {DDT_TRAIN}/images
      json_file: {DDT_TRAIN}/annotations.json

  val_data_sources:
    image_dir: {DDT_VAL}/images
    json_file: {DDT_VAL}/annotations.json

  num_classes: 2
  eval_class_ids: [1]

  batch_size: 2
  workers: 4
  pin_memory: true
  dataset_type: serialized

  augmentation:
    multi_scales: [640]
    train_spatial_size: [640, 640]
    eval_spatial_size: [640, 640]
    distortion_prob: 0.0
    iou_crop_prob: 0.8
    preserve_aspect_ratio: true

train:
  num_gpus: 1
  gpu_ids: [0]
  num_nodes: 1
  distributed_strategy: ddp

  num_epochs: 5
  validation_interval: 10
  checkpoint_interval: 10

  clip_grad_norm: 0.1
  precision: fp16
  activation_checkpoint: true
  enable_ema: true

  pretrained_model_path: {PASS2_CONTAINER}

  optim:
    optimizer: AdamW
    lr: 0.00005
    lr_backbone: 0.000005
    weight_decay: 0.0001
    lr_scheduler: MultiStep
    lr_decay: 0.1
    lr_steps: [70]
"""

# ------------------------------------------------------------
# 4. WRITE FILE
# ------------------------------------------------------------
PASS3_SPEC.write_text(
    pass3_yaml.strip() + "\n",
    encoding="utf-8"
    )



# ------------------------------------------------------------
# 5. DISPLAY RESULT
# ------------------------------------------------------------
print("============================================================")
print("PASS 3 CONFIG CREATED")
print("============================================================")
print("Config:")
print(PASS3_SPEC)

print("\nPass 2 pretrained checkpoint:")
print(PASS2_CONTAINER)

print("\nDDT train path:")
print(DDT_TRAIN)

print("\nDDT val path:")
print(DDT_VAL)

print("\n============================================================")
print("YAML PREVIEW")
print("============================================================")
print(pass3_yaml)


PASS 3 CONFIG CREATED
Config:
H:\Hobby Projrctrs\Sih26change\configs\rtdetr_step1_pass3_dronedetectionthesis_rgb.yaml

Pass 2 pretrained checkpoint:
/workspace/models/step1/step1_pass2_rtdetr_cradio_v3_l.pth

DDT train path:
/workspace/data/step1/dronedetectionthesis_rgb/tao_coco/train

DDT val path:
/workspace/data/step1/dronedetectionthesis_rgb/tao_coco/val

YAML PREVIEW

results_dir: /workspace/runs/step1_pass3

model:
  frozen_fm:
    enabled: true
    backbone: radio_v3-l
    checkpoint: /workspace/weights/c-radio-v3_l_half.pth.tar

  num_queries: 300
  num_select: 300
  num_feature_levels: 3
  return_interm_indices: [1, 2, 3]
  feat_strides: [8, 16, 32]
  feat_channels: [256, 256, 256]
  use_encoder_idx: [2]
  hidden_dim: 256
  nheads: 8
  dropout_ratio: 0.0
  enc_layers: 1
  dim_feedforward: 1024
  pe_temperature: 10000
  expansion: 1.0
  depth_mult: 1
  enc_act: gelu
  act: silu
  dec_layers: 6
  dn_number: 100
  eval_idx: -1

  vfl_loss_coef: 1.0
  bbox_loss_coef: 5.0
  giou_l

In [25]:
print("Docker run disabled by script")


PASS 3 DRY-RUN CONFIG CREATED
Config:
H:\Hobby Projrctrs\Sih26change\configs\rtdetr_step1_pass3_dryrun.yaml

COLAB STATUS
This notebook is running in the managed Colab environment.
The TAO Docker training command should be executed on the NVIDIA Docker host.

TAO image:
nvcr.io/nvidia/tao/tao-toolkit:7.2.0-pyt

DOCKER COMMAND
docker run --rm --gpus all --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 -v "H:\Hobby Projrctrs\Sih26change:/workspace" nvcr.io/nvidia/tao/tao-toolkit:7.2.0-pyt rtdetr train -e /workspace/configs/rtdetr_step1_pass3_dryrun.yaml


In [26]:
# ============================================================
# GOOGLE DRIVE CELL BYPASSED FOR LOCAL EXECUTION
# ============================================================
print('Google Drive logic bypassed.')


Google Drive logic bypassed.


In [27]:
# ============================================================
# SAVE ARTIFACTS (BYPASSED FOR LOCAL EXECUTION)
# ============================================================
print('Artifact saving bypassed.')


Artifact saving bypassed.


In [28]:
# ============================================================
# GOOGLE DRIVE CELL BYPASSED FOR LOCAL EXECUTION
# ============================================================
print('Google Drive logic bypassed.')


Google Drive logic bypassed.


In [29]:
# ============================================================
# GOOGLE DRIVE CELL BYPASSED FOR LOCAL EXECUTION
# ============================================================
print('Google Drive logic bypassed.')


Google Drive logic bypassed.


In [30]:
# ============================================================
# GOOGLE DRIVE CELL BYPASSED FOR LOCAL EXECUTION
# ============================================================
print('Google Drive logic bypassed.')


Google Drive logic bypassed.

In [31]:
# ============================================================
# GOOGLE DRIVE CELL BYPASSED FOR LOCAL EXECUTION
# ============================================================
print('Google Drive logic bypassed.')


Google Drive logic bypassed.


# PASS 4 — UAV-CB RGB

Use the visible images only.

The current public UAV-CB release contains 3,393 visible images at 1920×1080 with matching JSON annotations across building, cloud, ground, powerline and vegetation backgrounds.

Again, this notebook does not build or process the full test/inference pipeline.

In [32]:
# ============================================================
# 23. PASS 4 — DOWNLOAD / DISCOVER UAV-CB
# ============================================================

UAV_ROOT = STEP1_DATA / "uav-cb"
UAV_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

UAV_GDRIVE_ID = "1ojojmfWqKAFj-tpKt0kKlnc1cH805Ewp"

def find_uav_root(base):
    backgrounds = [
        "building",
        "cloud",
        "ground",
        "powerline",
        "vegetation"
    ]

    candidates = [
        base,
        base / "UAV-CB",
        base / "download",
        base / "extracted"
    ]

    candidates += [
        p.parent
        for p in base.rglob("visible")
        if p.name == "visible"
    ]

    for c in candidates:
        try:
            c = c.resolve()
        except Exception:
            continue

        if all(
            (c/bg/"visible").is_dir()
            for bg in backgrounds
        ):
            return c

    return None

UAV_DATA = find_uav_root(UAV_ROOT)

if UAV_DATA is None:
    subprocess.run(
        [sys.executable,"-m","pip","install","-q","-U","gdown"],
        check=True
    )

    import gdown

    DL = UAV_ROOT / "download"
    DL.mkdir(
        exist_ok=True
    )

    archive = DL / "UAV-CB.zip"

    if not archive.exists():
        gdown.download(
            id=UAV_GDRIVE_ID,
            output=str(archive),
            quiet=False,
            fuzzy=True
        )

    if not zipfile.is_zipfile(archive):
        raise RuntimeError(
            "The downloaded UAV-CB file is not a ZIP archive. "
            "Download the official archive manually if Google Drive returned a non-archive response."
        )

    EXTRACT = UAV_ROOT / "extracted"
    EXTRACT.mkdir(
        exist_ok=True
    )

    marker = EXTRACT / ".done"

    if not marker.exists():
        with zipfile.ZipFile(archive) as z:
            z.extractall(EXTRACT)

        marker.write_text(
            "ok",
            encoding="utf-8"
        )

    UAV_DATA = find_uav_root(EXTRACT)

print("UAV-CB data:", UAV_DATA)
assert UAV_DATA is not None


UAV-CB data: F:\Sih26change_Data\data\step1\uav-cb\extracted


In [33]:
# ============================================================
# 24. PASS 4 — UAV-CB RGB -> COCO
# ============================================================

backgrounds = [
    "building",
    "cloud",
    "ground",
    "powerline",
    "vegetation"
]

visible_files = []

for bg in backgrounds:
    visible_files.extend(
        sorted(
            (UAV_DATA/bg/"visible").glob("*.jpg")
        )
    )

print(
    "UAV-CB visible images:",
    len(visible_files)
)

assert visible_files

# Split without touching a huge test/inference pipeline.
# Keep 90% for training, 10% for lightweight validation.
rng = random.Random(42)
rng.shuffle(visible_files)

val_n = max(
    1,
    int(0.10*len(visible_files))
)

uav_val_files = visible_files[:val_n]
uav_train_files = visible_files[val_n:]

UAV_COCO = UAV_ROOT / "tao_coco"
UAV_TRAIN = UAV_COCO / "train"
UAV_VAL = UAV_COCO / "val"

(UAV_TRAIN/"images").mkdir(
    parents=True,
    exist_ok=True
)

(UAV_VAL/"images").mkdir(
    parents=True,
    exist_ok=True
)

def uav_boxes(json_path):
    d = json.loads(
        json_path.read_text(
            encoding="utf-8",
            errors="replace"
        )
    )

    boxes = []

    for shape in d.get(
        "shapes",
        []
    ):

        if str(
            shape.get(
                "label",
                ""
            )
        ).strip().lower() != "drone":
            continue

        pts = shape.get(
            "points",
            []
        )

        if len(pts) < 2:
            continue

        xs = [
            float(p[0])
            for p in pts
        ]

        ys = [
            float(p[1])
            for p in pts
        ]

        x1,x2 = min(xs),max(xs)
        y1,y2 = min(ys),max(ys)

        boxes.append([
            x1,y1,x2-x1,y2-y1
        ])

    return boxes

def build_uav(files,image_dir,json_path,description):
    images = []
    annotations = []
    ann_id = 1

    for image_id,src in enumerate(
        sorted(files),
        start=1
    ):

        dst = image_dir / src.name

        if dst.exists():
            dst = image_dir / (
                f"{src.parent.parent.name}__{src.name}"
            )

        shutil.copy2(
            src,
            dst
        )

        with Image.open(src) as im:
            width,height = im.size

        images.append({
            "id":image_id,
            "file_name":dst.name,
            "width":width,
            "height":height
        })

        for bbox in uav_boxes(
            src.with_suffix(".json")
        ):

            x,y,w,h = map(
                float,
                bbox
            )

            x = max(
                0,min(x,width)
            )

            y = max(
                0,min(y,height)
            )

            w = max(
                0,min(w,width-x)
            )

            h = max(
                0,min(h,height-y)
            )

            if w <= 0 or h <= 0:
                continue

            annotations.append({
                "id":ann_id,
                "image_id":image_id,
                "category_id":1,
                "bbox":[x,y,w,h],
                "area":w*h,
                "iscrowd":0
            })

            ann_id += 1

    payload = {
        "info":{"description":description},
        "licenses":[],
        "images":images,
        "annotations":annotations,
        "categories":[{"id":1,"name":"drone"}]
    }

    json_path.write_text(
        json.dumps(payload),
        encoding="utf-8"
    )

    return len(images),len(annotations)

print(
    "Train:",
    build_uav(
        uav_train_files,
        UAV_TRAIN/"images",
        UAV_TRAIN/"annotations.json",
        "SIH26247 Pass 4 UAV-CB RGB training"
    )
)

print(
    "Val:",
    build_uav(
        uav_val_files,
        UAV_VAL/"images",
        UAV_VAL/"annotations.json",
        "SIH26247 Pass 4 UAV-CB RGB validation"
    )
)


UAV-CB visible images:

 3393


Train: (3054, 3054)


Val: (339, 339)


In [34]:
# ============================================================
# 25. PASS 4 — CONFIG FROM PASS 3 CHECKPOINT
# ============================================================

PASS4_SPEC = CONFIG_ROOT / "rtdetr_step1_pass4_uav_cb_rgb.yaml"

PASS3_CONTAINER = "/workspace/models/step1/step1_pass3_rtdetr_cradio_v3_l.pth"

pass4_yaml = pass3_yaml

pass4_yaml = pass4_yaml.replace(
    "results_dir: /workspace/runs/step1_pass3",
    "results_dir: /workspace/runs/step1_pass4"
)

pass4_yaml = pass4_yaml.replace(
    "/workspace/data/step1/dronedetectionthesis_rgb/tao_coco",
    "/workspace/data/step1/uav-cb/tao_coco"
)

pass4_yaml = pass4_yaml.replace(
    PASS2_CONTAINER,
    PASS3_CONTAINER
)

PASS4_SPEC.write_text(
    pass4_yaml,
    encoding="utf-8"
    )



print(PASS4_SPEC)


H:\Hobby Projrctrs\Sih26change\configs\rtdetr_step1_pass4_uav_cb_rgb.yaml


In [35]:
print("Docker run disabled by script")


$ docker run --rm --gpus all --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 -v "H:\Hobby Projrctrs\Sih26change:/workspace" nvcr.io/nvidia/tao/tao-toolkit:7.2.0-pyt rtdetr train -e /workspace/configs/rtdetr_step1_pass4_dryrun.yaml



=== TAO Toolkit PyTorch ===

NVIDIA Release 7.2.0-PyTorch (build 286725635)
TAO Toolkit Version 7.2.0

Various files include modifications (c) NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

This container image and its contents are governed by the TAO Toolkit End User License Agreement.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/tao-toolkit-software-license-agreement

  Using driver version 592.82 which has support for CUDA 13.1.  This container
  was built with CUDA 13.2 and will be run in Minor Version Compatibility mode.
  CUDA Forward Compatibility is preferred over Minor Version Compatibility for use
  with this container but was unavailable:
  [[]]
  See https://docs.nvidia.com/deploy/cuda-compatibility/ for details.

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.w

In [36]:
print("Docker run disabled by script")


$ docker run --rm --gpus all --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 -v "H:\Hobby Projrctrs\Sih26change:/workspace" nvcr.io/nvidia/tao/tao-toolkit:7.2.0-pyt rtdetr train -e /workspace/configs/rtdetr_step1_pass4_uav_cb_rgb.yaml



=== TAO Toolkit PyTorch ===

NVIDIA Release 7.2.0-PyTorch (build 286725635)
TAO Toolkit Version 7.2.0

Various files include modifications (c) NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

This container image and its contents are governed by the TAO Toolkit End User License Agreement.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/tao-toolkit-software-license-agreement

  Using driver version 592.82 which has support for CUDA 13.1.  This container
  was built with CUDA 13.2 and will be run in Minor Version Compatibility mode.
  CUDA Forward Compatibility is preferred over Minor Version Compatibility for use
  with this container but was unavailable:
  [[]]
  See https://docs.nvidia.com/deploy/cuda-compatibility/ for details.

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.w

# Storage / handoff rule

You do **not** need to retain every dataset simultaneously.

After a pass succeeds and its checkpoint is safely copied:

```text
Pass 1 complete
    ↓
keep Pass-1 checkpoint
    ↓
delete Seraphim raw/converted data if storage is needed

Pass 2 complete
    ↓
keep Pass-2 checkpoint
    ↓
delete RealDroneVision raw/converted data if storage is needed

Pass 3 complete
    ↓
keep Pass-3 checkpoint
    ↓
delete DDT RGB extracted/converted data if storage is needed

Pass 4 complete
    ↓
keep final Step-1 checkpoint
```

Keep these permanent model artifacts:

```text
step1_pass1_rtdetr_cradio_v3_l.pth
step1_pass2_rtdetr_cradio_v3_l.pth
step1_pass3_rtdetr_cradio_v3_l.pth
step1_rgb_final_rtdetr_cradio_v3_l.pth
```

No full prediction corpus is created by this notebook.

In [37]:
# ============================================================
# 28. OPTIONAL: CLEAN A COMPLETED PASS
# ============================================================
# Run ONLY after checking that the corresponding checkpoint exists.
#
# Examples:
#
# shutil.rmtree(REAL_ROOT, ignore_errors=True)   # after Pass 2
# shutil.rmtree(DDT_ROOT, ignore_errors=True)    # after Pass 3
# shutil.rmtree(UAV_ROOT, ignore_errors=True)    # after Pass 4
#
# The checkpoint files under STEP1_MODELS are NOT deleted.


## Docker command used for TAO 7.2 on the NVIDIA host

The actual training host uses the same container that already worked for your Pass 1:

```bash
nvcr.io/nvidia/tao/tao-toolkit:7.2.0-pyt
```

and the recommended runtime flags:

```text
--gpus all
--ipc=host
--ulimit memlock=-1
--ulimit stack=67108864
```

For your Windows Docker setup, the mounted project contains:

```text
/workspace
/workspace/data
/workspace/runs
/workspace/models
```

The notebook's TAO YAML paths are written against those container paths.

No ONNX conversion or inference is needed between Pass 2, 3, and 4.